# Experiment 4.4.3 — Heterogeneous Stage-2 membrane timescales

Analysis-only notebook. It compares `single250`, `single500`, `tau250_500`, and `tau125_250_500` under the same Dense-RSNN architecture and readout matrix.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'AGENTS.md').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('Could not find writingRing repository root')

REPO_ROOT = find_repo_root()
ARTIFACT_DIR = REPO_ROOT / 'notebooks' / 'artifacts' / 'experiment_4_4_3_multitau_memory' / 'stage2_multitau_dense_v1'
EXPECTED_SEEDS = (11, 23, 37, 53, 71)
EXPECTED_PROFILES = ('single250', 'single500', 'tau250_500', 'tau125_250_500')
required = ('runs.csv', 'summary.csv', 'paired_effects.csv', 'paired_effects_summary.csv', 'manifest.json')
missing = [name for name in required if not (ARTIFACT_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f'Missing finalized Exp4.4.3 artifacts: {missing}')

runs = pd.read_csv(ARTIFACT_DIR / 'runs.csv')
summary = pd.read_csv(ARTIFACT_DIR / 'summary.csv')
effects = pd.read_csv(ARTIFACT_DIR / 'paired_effects.csv')
effects_summary = pd.read_csv(ARTIFACT_DIR / 'paired_effects_summary.csv')
manifest = json.loads((ARTIFACT_DIR / 'manifest.json').read_text())

test = runs[runs['split'] == 'test']
assert set(test['seed']) == set(EXPECTED_SEEDS)
assert set(test['profile']) == set(EXPECTED_PROFILES)
assert len(test) == len(EXPECTED_SEEDS) * len(EXPECTED_PROFILES)
manifest


## Profile definitions and aggregate results
The heterogeneous profiles change only fixed Stage-2 beta/tau assignment; trainable parameter count is unchanged.


In [ ]:
cols = [
    'profile', 'is_heterogeneous', 'tau_profile_ms', 'group_widths',
    'uend_linear_ba_mean', 'uend_linear_ba_std',
    'hidden_whole_count_linear_ba_mean', 'hidden_whole_count_linear_ba_std',
    'output_whole_count_ba_mean', 'output_whole_count_ba_std',
    'uend_minus_hidden_count_ba_mean', 'uend_minus_output_count_ba_mean',
]
display(summary[cols].sort_values('profile').reset_index(drop=True))


## Endpoint memory quality
Primary question: does either heterogeneous profile exceed the homogeneous `single250` Uend memory reference?


In [ ]:
order = list(EXPECTED_PROFILES)
frame = summary.set_index('profile').loc[order]
x = np.arange(len(order))
fig, ax = plt.subplots(figsize=(9.5, 5.2))
ax.errorbar(x, frame['uend_linear_ba_mean'], yerr=frame['uend_linear_ba_std'], marker='o', linestyle='none', capsize=5)
ax.set_xticks(x, order, rotation=15)
ax.set_ylabel('Test balanced accuracy')
ax.set_ylim(0, 0.7)
ax.set_title('Stage-2 endpoint memory: Uend + Linear')
ax.grid(axis='y', alpha=0.25)
plt.show()


## Paired gain versus single250
This is the primary architecture comparison because `single250` was the best homogeneous Dense-RSNN endpoint-memory condition in Exp4.4.1.


In [ ]:
primary = effects_summary[(effects_summary['baseline'] == 'single250') & (effects_summary['readout'] == 'uend_linear')].copy()
display(primary.sort_values('profile').reset_index(drop=True))
fig, ax = plt.subplots(figsize=(8.5, 5.2))
x = np.arange(len(primary))
ax.errorbar(x, primary['mean'], yerr=primary['std'], marker='o', linestyle='none', capsize=5)
ax.set_xticks(x, primary['profile'], rotation=15)
ax.axhline(0, linewidth=1)
ax.set_ylabel('Paired delta BA vs single250')
ax.set_title('Does heterogeneous tau improve Uend memory?')
ax.grid(axis='y', alpha=0.25)
plt.show()


## Full readout matrix
A profile can improve Uend memory without improving spike-accessible count readouts. Keep all three levels visible.


In [ ]:
readouts = {
    'Uend + Linear': ('uend_linear_ba_mean', 'uend_linear_ba_std'),
    'HiddenCount + Linear': ('hidden_whole_count_linear_ba_mean', 'hidden_whole_count_linear_ba_std'),
    'Output WholeCount': ('output_whole_count_ba_mean', 'output_whole_count_ba_std'),
}
fig, ax = plt.subplots(figsize=(10, 5.5))
offsets = np.linspace(-0.18, 0.18, len(readouts))
for offset, (label, (mean_col, std_col)) in zip(offsets, readouts.items()):
    ax.errorbar(x + offset, frame[mean_col], yerr=frame[std_col], marker='o', linestyle='none', capsize=3, label=label)
ax.set_xticks(x, order, rotation=15)
ax.set_ylabel('Test balanced accuracy')
ax.set_ylim(0, 0.7)
ax.set_title('Readout matrix across Stage-2 tau profiles')
ax.legend()
ax.grid(axis='y', alpha=0.25)
plt.show()


## State-to-spike information gap
A larger positive Uend-to-count gap means the architecture stores useful endpoint information that is not yet efficiently expressed by spikes.


In [ ]:
gap_cols = ['profile', 'uend_minus_hidden_count_ba_mean', 'uend_minus_output_count_ba_mean']
display(frame.reset_index()[gap_cols])
fig, ax = plt.subplots(figsize=(9.5, 5.2))
ax.plot(np.arange(len(order)), frame['uend_minus_hidden_count_ba_mean'], marker='o', label='Uend - HiddenCount')
ax.plot(np.arange(len(order)), frame['uend_minus_output_count_ba_mean'], marker='o', label='Uend - OutputCount')
ax.axhline(0, linewidth=1)
ax.set_xticks(np.arange(len(order)), order, rotation=15)
ax.set_ylabel('Balanced-accuracy gap')
ax.set_title('State-to-spike accessibility gap')
ax.legend()
ax.grid(axis='y', alpha=0.25)
plt.show()


## Firing and tail diagnostics
Use these to distinguish a useful multi-timescale state from a configuration that merely increases persistent activity.


In [ ]:
diag_cols = [
    'profile',
    'state_events_per_neuron_second_mean', 'output_events_per_neuron_second_mean',
    'state_tail_event_fraction_mean', 'output_tail_event_fraction_mean',
]
display(frame.reset_index()[diag_cols])
fig, ax = plt.subplots(figsize=(9.5, 5.2))
ax.plot(np.arange(len(order)), frame['state_tail_event_fraction_mean'], marker='o', label='Stage-2 tail')
ax.plot(np.arange(len(order)), frame['output_tail_event_fraction_mean'], marker='o', label='Output tail')
ax.set_xticks(np.arange(len(order)), order, rotation=15)
ax.set_ylabel('Tail-event fraction')
ax.set_title('Post-end activity by tau profile')
ax.legend()
ax.grid(axis='y', alpha=0.25)
plt.show()


## Decision rule
Prefer a heterogeneous profile only if the paired Uend gain over `single250` is positive and reasonably seed-consistent. Treat Output WholeCount as a secondary deployment-facing metric during this architecture phase; a larger Uend-to-count gap identifies later state-to-spike expression work rather than invalidating the memory architecture.
